# Wang 5-Stack CNN training — 3 classes

Trains the Wang et al. 5-Stack CNN baseline for 3-class rain intensity classification.

Classes:
- `0`: light
- `1`: moderate
- `2`: intense

Adaptations from the original paper:
- Output layer `Dense(3, softmax)` instead of `Dense(5, softmax)`;
- Input `(256, 256, 1)`: 256×256 grayscale spectrograms;
- Same data pipeline, focal loss with γ = 2.0, optimizer and callbacks as `ArquiteturaCNN3class` for a fair comparison;
- Best model saved to `best_model_wang_3class.keras`

In [ ]:
import gc
import random
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D, LeakyReLU, BatchNormalization,
    MaxPooling2D, SpatialDropout2D,
    GlobalAveragePooling2D, Dense,)
from tensorflow.keras.regularizers import l2
from Multiclasse_data_pipeline import build_datasets, configure_gpu
from tensorflow.keras import backend as K

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Experiment parameters, same as ArquiteturaCNN3class
CSV_PATH       = "Split70-15-15_3class.csv"
N_CLASSES      = 3
EPOCHS         = 50
BATCH_SIZE     = 64
LEARNING_RATE  = 1e-3
SEED           = 42

PATIENCE_EARLY_STOPPING = 15
PATIENCE_REDUCE_LR      = 5
LR_REDUCE_FACTOR        = 0.5
LR_MIN                  = 1e-6


In [ ]:
class GarbageCollectionCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        collected = gc.collect()
        print(f"   [GC] {collected} objetos liberados.")


In [ ]:


def categorical_focal_loss(gamma=2.0, alpha=None):
    """
    Categorical focal loss (Lin et al., 2017).
    - gamma: focusing parameter for hard examples.
    - alpha: per-class weights; None = uniform.
    """
    if alpha is not None:
        alpha_tensor = tf.constant(alpha, dtype=tf.float32)
    else:
        alpha_tensor = None

    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())
        ce = -y_true * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1.0 - p_t, gamma)
        focal_ce = focal_weight * ce
        if alpha_tensor is not None:
            focal_ce = focal_ce * alpha_tensor
        return tf.reduce_sum(focal_ce, axis=-1)

    return loss


In [ ]:
# Wang et al. 5-Stack CNN with 3-class output: Dense(3) instead of Dense(5)
def build_wang_5stack_cnn(input_shape=(256, 256, 1), n_classes=N_CLASSES):
    model = Sequential(name="Wang_5Stack_CNN_3class")

    # L1 — Conv 7×7, stride 2, 64 filters
    model.add(Conv2D(64, (7, 7), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005), input_shape=input_shape))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(SpatialDropout2D(0.07))

    # L2 — Conv 5×5, stride 2, 48 filters + MaxPool
    model.add(Conv2D(48, (5, 5), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(SpatialDropout2D(0.07))

    # L3 — Conv 5×5, stride 2, 48 filters + MaxPool
    model.add(Conv2D(48, (5, 5), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(SpatialDropout2D(0.07))

    # L4 — Conv 3×3, stride 2, 32 filters
    model.add(Conv2D(32, (3, 3), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())
    model.add(SpatialDropout2D(0.14))

    # L5 — Conv 3×3, stride 2, 64 filters
    model.add(Conv2D(64, (3, 3), strides=(2, 2), padding='same',
                     kernel_regularizer=l2(0.0005)))
    model.add(LeakyReLU(alpha=0.1))
    model.add(BatchNormalization())

    # Output — GAP + Dense(3, softmax)
    model.add(GlobalAveragePooling2D())
    model.add(Dense(n_classes, activation='softmax'))

    return model


In [ ]:
# Data loading, model construction and compilation
configure_gpu()

train_ds, val_ds, test_ds = build_datasets(
    CSV_PATH,
    batch_size=BATCH_SIZE,
    label_mode="one_hot",
    augment_train=True,
    mixup_train=True,)

model = build_wang_5stack_cnn(input_shape=(256, 256, 1), n_classes=N_CLASSES)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=categorical_focal_loss(gamma=2.0, alpha=[0.4, 0.4, 0.2]),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy"),
    ],
)

model.summary()


In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model_wang_3class.keras",
        monitor="val_loss",
        mode="min",
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        mode="min",
        patience=PATIENCE_EARLY_STOPPING,
        restore_best_weights=True,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=LR_REDUCE_FACTOR,
        patience=PATIENCE_REDUCE_LR,
        min_lr=LR_MIN,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(filename="training_history_wang_3class.csv"),
    GarbageCollectionCallback(),]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)


In [ ]:
# Training summary
n_epochs_run     = len(history.history["loss"])
best_val_loss    = min(history.history["val_loss"])
best_val_acc_idx = int(np.argmax(history.history["val_accuracy"]))
best_val_acc     = history.history["val_accuracy"][best_val_acc_idx]

print(f"Épocas executadas:   {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:     {best_val_loss:.4f}")
print(f"Melhor val_accuracy: {best_val_acc:.4f} (época {best_val_acc_idx + 1})")
print(f"Modelo salvo em:     best_model_wang_3class.keras")
print(f"Histórico em:        training_history_wang_3class.csv")


In [ ]:
# Learning curves

# Loss
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.title("Loss — Wang 5-Stack CNN (Focal Loss γ=2.0 + Mixup)")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()

# Accuracy
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="train")
plt.plot(history.history["val_accuracy"], label="val")
plt.title("Accuracy — Wang 5-Stack CNN")
plt.xlabel("Época")
plt.ylabel("Accuracy")
plt.ylim([0, 1])
plt.legend()
plt.tight_layout()
plt.show()

print("=" * 40)
print("RESUMO FINAL")
print("=" * 40)
print(f"Épocas executadas:   {n_epochs_run} / {EPOCHS}")
print(f"Melhor val_loss:     {best_val_loss:.4f}")
print(f"Melhor val_accuracy: {best_val_acc:.4f} (época {best_val_acc_idx + 1})")
print("=" * 40)
print()
print("PRÓXIMO PASSO: rode o Wang_Evaluation_3class.ipynb para as métricas finais.")
